# Image → 3D Scene (Stage B · TripoSR)

Generates a real 3D mesh per detected object (TripoSR), placed on a fitted ground plane → `scene.glb`.

**Before running:** `Runtime → Change runtime type → T4 GPU`. If you ran an earlier attempt, also `Runtime → Disconnect and delete runtime` first (a previous TripoSR `requirements.txt` install corrupts numpy). Then run top to bottom.

In [ ]:
# 0. Confirm GPU
!nvidia-smi -L

## 1. Install TripoSR (curated — NOT its requirements.txt)
TripoSR's `requirements.txt` pins ancient numpy/transformers/hf-hub that break Colab. We install only what's needed, keeping Colab's modern numpy.

In [ ]:
import sys
![ -d /content/TripoSR ] || git clone https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR
%cd /content/TripoSR
# Minimal deps for `import tsr.system` + extract_mesh (no requirements.txt, no rembg).
!pip install -q einops omegaconf jaxtyping typeguard moderngl trimesh
!pip install -q --no-build-isolation "git+https://github.com/tatsy/torchmcubes.git"
# Our detection / depth / geometry deps (modern; must NOT downgrade numpy).
!pip install -q "transformers>=4.44,<5" "huggingface_hub>=0.24" accelerate timm scipy xatlas
if '/content/TripoSR' not in sys.path:
    sys.path.insert(0, '/content/TripoSR')
# Sanity: numpy must be intact AND TripoSR must import.
import numpy as np; print('numpy', np.__version__)
from tsr.system import TSR  # noqa: F401
print('OK: TripoSR imports')

## 2. Get the pipeline code

In [ ]:
REPO = '/content/image-3d-pipeline'
BRANCH = 'claude/sweet-cori-kzkhyr'
![ -d {REPO} ] || git clone --branch {BRANCH} https://github.com/sanjanamani/image-3d-pipeline.git {REPO}
!cd {REPO} && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('pipeline code ready:', REPO)

## 3. Upload your photo

In [ ]:
from google.colab import files
uploaded = files.upload()
IMAGE_PATH = '/content/' + next(iter(uploaded))
print('uploaded:', IMAGE_PATH)

## 4. Build the 3D scene (TripoSR per object)

In [ ]:
import os
os.chdir(REPO)
from scene_build import run_build
glb_path = run_build(IMAGE_PATH, output_dir='/content/outputs', backend='triposr')
print('\nDONE ->', glb_path)

## 5. View inline + download

In [ ]:
import base64
from IPython.display import HTML, display

b64 = base64.b64encode(open(glb_path, 'rb').read()).decode()
display(HTML(f'''
<script type="module" src="https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js"></script>
<model-viewer src="data:model/gltf-binary;base64,{b64}"
  camera-controls auto-rotate shadow-intensity="1"
  style="width:100%;height:520px;background:#222;"></model-viewer>
'''))

from google.colab import files as _f
_f.download(glb_path)